Practica 9: Mediante el fichero Datos Pedidos.xlsx, cargar en un formulario de tkinter, solamente los pedidos según el pais indicado en la caja de texto de filtro. (El formulario, inicialmente cargara todos los registros y despues en un entry indicaremos el nombre del Pais a filtrar). Mediante un botón visualizar las filas que cumplan la condición del valor de entry (alias caja de texto)

In [ ]:
#Solución practica 9
#0. Importar las librerias necesarias
import pandas as pd
import tkinter as tk            #Trabajar con el formulario
from tkinter import ttk         #Trabajar con la tabla

#1. Ajustar la ruta del excel
ruta_excel = r"C:\PCAD\Datos Pedidos.xlsx"

#2. Cargar el dataframe con los datos de los importes por franja
pedidos = pd.read_excel(ruta_excel, sheet_name="Datos")

#3. Crear la ventana principal
root = tk.Tk()
root.title("Analisis Franjas")

#4. Guardar dimensiones en variables
ancho_ventana = 500
alto_ventana = 300
ancho_pantalla = root.winfo_screenwidth()
alto_pantalla = root.winfo_screenheight()

#5. Calcular la posicion de la izquierda (x) y de arriba (y)
pos_x =(ancho_pantalla // 2) - (ancho_ventana // 2)
pos_y =(alto_pantalla // 2) - (alto_ventana // 2)

#6. Geometry root.geometry("500x300+100+50")
root.geometry(f"{ancho_ventana}x{alto_ventana}+{pos_x}+{pos_y}")

#7. Crear el frame para la tabla
frm = tk.LabelFrame(root, text="Pedidos", padx=5, pady=5)
frm.pack(side="left", padx=5, pady=5, fill="both", expand=True)

#8. Crear el frame para la caja de texto (entry) + el button
frm2 = tk.LabelFrame(root, text="Filtro", padx=5, pady=5)
frm2.pack(side="right", padx=5, pady=5, fill="y")

#9. Crear un widget Treeview (lo tenemos dentro de la clase ttk, la tendremos que llamar primero)
tabla = ttk.Treeview(root)   #Widget treeview
tabla.pack(fill="both")

#10. Definir columnas, con la lista de nombres del dataframe
tabla['columns'] = list(pedidos.columns)
tabla['show'] = "headings"

#11. Cabeceras
for columna in pedidos.columns:
    tabla.heading(columna, text=columna)
    tabla.column(columna, width=120)

#12. Insertar las filas
for _, fila in pedidos.iterrows():
    tabla.insert("","end",values=list(fila))

#13. Crear el scroll vertical
scroll_y = ttk.Scrollbar(root,orient="vertical",command=tabla.yview)

#14. Crear el scroll horizontal
scroll_x = ttk.Scrollbar(root,orient="horizontal",command=tabla.xview)

#15. Conectar los scrolls a la tabla
tabla.configure(yscrollcommand=scroll_y.set,
                xscrollcommand=scroll_x.set)

#16. Posicionar + visualizar con el pack
scroll_y.pack(side="right", fill="y")
scroll_x.pack(side="bottom", fill="x")

#17. Crear la caja de texto con entry
entry = tk.Entry(frm2)
entry.pack(pady=5)

#18. Crear la función para el filtrado
def filtrar_tabla():
    #Primero mirar si hay valor en el entry o caja de texto
    #Buscar el valor de entry dentro de un conjunto set con los valores unique
    #Como eliminar la tabla, en primer lugar
    tabla.delete(*tabla.get_children())
    #Capturar el pais a filtrar
    pais_a_filtrar = entry.get().upper()
    #Filtrado de dataframe
    pedidos_filtrado = pedidos[pedidos['Pais'].str.upper()==pais_a_filtrar]
    #Insertar las filas
    for _, fila in pedidos_filtrado.iterrows():
        tabla.insert("","end",values=list(fila))

#19. Crear el botón para realizar el filtro
tk.Button(frm2, text="Filtrar por Pais", command=filtrar_tabla).pack(pady=5)

#12. Mostrar la ventana
root.mainloop()

Trabajar con los graficos de Python:  
4 Librerias para trabajar con graficos;
- matplotlib: Librería base para graficos en Python, flexible y precisa, ideal para control total. El codigo puede ser largo
- seaborn: Libreria construida sobre matplotlib. Facilita gráficos estadísticos
con mejores estilos y menos codigo
- bokeh: Pensada para graficos interactivos en navegador. Permite zoom, tooltips, dashboard sin usar java
- streamlit: para mostrar resultados interactivos.

Cargaremos en un dataframe los Datos Pedidos en excel, averiguaremos la suma de importe por pais y haremos un grafico.

In [ ]:
#0. Importar las librerias necesarias
import pandas as pd

#1. Ajustar la ruta del excel
ruta_excel = r"C:\PCAD\Datos Pedidos.xlsx"

#2. Cargar el dataframe con los datos
pedidos = pd.read_excel(ruta_excel, sheet_name="Datos")

#3. Crear la agrupación por Pais + la suma de Importe
pedidos.groupby(['Pais'])['Importe'].sum()

#Que nos devuelve este codigo, devuelve una serie
type(pedidos.groupby(['Pais'])['Importe'].sum())

#Como obtener varias metricas de una dimension, con agg
pedidos.groupby(['Pais'])['Importe'].agg(['sum','mean','count'])

#Como convertir esta agrupación en un dataframe correcto, con reset_index al final
pedidos.groupby(['Pais'])['Importe'].agg(['sum','mean','count']).reset_index()

#Trabajar con una referencia a este dataframe
pedidos_agrupados = pedidos.groupby(['Pais'])['Importe'].agg(['sum','mean','count']).reset_index()

Agregaciones:  
- sum = para sumar  
- mean = para promedio  
- max = para el valor mas grande  
- min = para el valor mas pequeño  
- count = para contar

In [ ]:
#Grafico de barras o columnas, el concepto solo quiero ver los 5 paises que mas importe tienen. con nlargest veremos los n elementos mas grandes
pedidos_agrupados_top5 = pedidos_agrupados.nlargest(5, "sum")

In [ ]:
pedidos_agrupados_top5

In [ ]:
#Para usar matplotlib, debo instalar la libreria externa
#desde cmd >pip install matplotlib
#0. Llamar a la libreria
import matplotlib.pyplot as plt

#1. Para crear un grafico, debo reservar espacio en el kernel con figure
plt.figure(figsize=(12,6))

#2. Crear el grafico de columnas tipo bar, kind='bar'
pedidos_agrupados_top5.plot(kind='bar', x='Pais', y='sum', color='green', label="Importe")

#3. Ajustar parametros del grafico
plt.title("Top 5 - Importe por pais")    #el titulo del grafico
plt.xlabel("Paises")                     #El titulo del eje de las x
plt.ylabel("Suma Importe")
plt.legend()
plt.xticks(rotation=45)

#4. mostrar el grafico plt.show()
plt.show()

In [ ]:
#Aprovechando el top 5, vamos a ver un grafico circular con kind = 'pie'
#0. Llamar a la libreria
import matplotlib.pyplot as plt

#1. Para crear un grafico, debo reservar espacio en el kernel con figure
plt.figure(figsize=(8,8))

#2. Crear el grafico
pedidos_agrupados_top5.plot(kind='pie', y='sum', labels=pedidos_agrupados_top5['Pais'], autopct='%1.2f%%')

#3. Anclar la leyenda a la parte derecha
plt.legend(title="Paises", bbox_to_anchor=(1.05,1))

#3. Mostrar el gráfico
plt.show()

Practica 10: Mediante el dataframe de pedidos agrupados, ver en un grafico circular los 3 paises que menos importe tienen, con sus % reales.
Si en el dataframe el % de un pais es el 15,38%, en el grafico tengo que ver
ese mismo porcentaje.
Pista!!! Si queremos los 3 que menos, necesitaremos 4 porciones.